# 心思维 AI 扒谱助手｜统一运行入口

**Notebook：** `XinsW_Music_Transcriber_Main.ipynb`  
**用途：** 办公室与家中统一使用的 Colab 入口。代码以 GitHub 指定分支为准，模型与测试数据以 Google Drive 为准。

## 安全约定
- 本 Notebook 的初始版本只做环境与文件盘点，不安装依赖、不下载模型、不启动推理、不删除或覆盖文件。
- 每次运行先记录 Python、GPU、关键包版本及 Drive 目录现状。
- 只有在核对旧 Notebook 的成功运行代码后，才迁移模型加载与推理步骤。
- 人声分离结果可能残留混响或合唱；模型评估必须标注输入音频来源与质量，不能默认视为干净独唱。


## 01｜连接 Google Drive

运行下面单元格后按提示授权。不会修改 Drive 文件。


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
print('Drive mounted:', MY_DRIVE.exists())
print('MyDrive:', MY_DRIVE)


## 02｜运行环境盘点

只读取当前运行时信息；不安装、不升级任何软件。


In [ ]:
import sys, platform, importlib.metadata as md
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('CUDA runtime:', torch.version.cuda)
except Exception as exc:
    print('PyTorch check:', type(exc).__name__, str(exc))

packages = ['numpy', 'librosa', 'torchcrepe', 'crepe', 'vocalparse', 'transformers', '音频']
for package in packages:
    try:
        print(f'{package}:', md.version(package))
    except md.PackageNotFoundError:
        print(f'{package}: not installed / distribution name not found')


## 03｜Drive 现有目录盘点

仅列出 MyDrive 顶层目录名称，帮助定位已有模型与测试数据；不遍历文件内容、不移动、不删除。


In [ ]:
if not MY_DRIVE.exists():
    raise FileNotFoundError('Google Drive 尚未挂载，请先运行 01 单元格。')
for item in sorted(MY_DRIVE.iterdir(), key=lambda p: p.name.lower()):
    kind = 'DIR ' if item.is_dir() else 'FILE'
    print(kind, item.name)


## 04｜后续模块（盘点完成后再启用）

1. GitHub 分支与代码版本核对  
2. 现有模型文件定位与校验（不重复下载）  
3. VocalParse 主流程接入  
4. TorchCREPE 交叉验证接入  
5. Qwen3-ASR 歌词识别接入  
6. Music Event / MusicXML 输出与回归测试  

**当前状态：** 环境盘点骨架。尚未迁移旧 Notebook 的推理代码；不得将本文件视为已完成模型运行验证。


## 05｜统一授权并连接项目代码

本步骤从 Colab Secrets 读取统一保存的 `GH_TOKEN`，再获取 GitHub 私有仓库指定分支。Token 不写入 Notebook、命令行参数或 Git 持久配置。家里与办公室使用同一个 Google 账户和这份主 Notebook；首次设置 Secret 并允许本 Notebook 访问即可。已有目录不会删除、覆盖或自动切换分支。

In [ ]:
from pathlib import Path
import base64
import json
import os
import shutil
import subprocess
import sys
import urllib.error
import urllib.request

REPO_DIR = Path("/content/xinsw-music-transcriber")
STAGING_DIR = Path("/content/.xinsw-music-transcriber-clone")
FAILED_DIR = Path("/content/.xinsw-music-transcriber-clone-failed")
REPO_URL = "https://github.com/Trifida/xinsw-music-transcriber.git"
REPO_API = "https://api.github.com/repos/Trifida/xinsw-music-transcriber"
REPO_BRANCH = "feature/vocalparse-event-musicxml"

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(
            f"目标目录已存在但不是 Git 仓库：{REPO_DIR}。"
            "为保护现有文件，Notebook 不会删除或覆盖它。"
        )
    current_branch = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
    ).strip()
    if current_branch != REPO_BRANCH:
        raise RuntimeError(
            f"现有仓库分支为 {current_branch!r}，预期为 {REPO_BRANCH!r}。"
            "为保护现有工作，Notebook 不会自动切换分支。"
        )
    print("发现已有项目仓库，保留原样：", REPO_DIR)
else:
    from google.colab import userdata
    try:
        gh_token = userdata.get("GH_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "尚未取得 Colab Secret：GH_TOKEN。请在 Colab 左侧 Secrets 中添加该名称，"
            "并允许本主 Notebook 访问。不要把 Token 粘贴到代码或聊天中。"
        ) from exc
    if not gh_token:
        raise RuntimeError("Colab Secret GH_TOKEN 为空，请检查 Secret 设置。")

    # 先用 GitHub API 验证 Token 是否确实能读取这个私有仓库，不显示 Token。
    api_request = urllib.request.Request(
        REPO_API,
        headers={"Authorization": f"Bearer {gh_token}", "Accept": "application/vnd.github+json"}
    )
    try:
        with urllib.request.urlopen(api_request, timeout=30) as response:
            repo_info = json.loads(response.read().decode("utf-8"))
        print("GitHub 私有仓库读取验证通过：", repo_info.get("full_name", "仓库已识别"))
    except urllib.error.HTTPError as exc:
        del gh_token
        if exc.code == 404:
            raise RuntimeError(
                "GitHub 返回 404：当前 Token 看不到该私有仓库。请检查创建 Token 时是否选中了 "
                "Trifida/xinsw-music-transcriber，以及 Resource owner 是否正确。"
            ) from None
        if exc.code == 403:
            raise RuntimeError(
                "GitHub 返回 403：Token 可能尚未获准访问该仓库，或组织要求额外批准。"
                "请检查 Token 的仓库访问范围和组织授权。"
            ) from None
        raise RuntimeError(f"GitHub 仓库读取验证失败，HTTP {exc.code}。") from None
    except Exception as exc:
        del gh_token
        raise RuntimeError(f"GitHub 仓库读取验证未完成：{type(exc).__name__}: {exc}") from None

    # 上一步已验证 PAT 可读仓库。PAT 使用 Git HTTPS Basic：GitHub 登录名:Token。
    auth_value = "AUTHORIZATION: basic " + base64.b64encode(
        ("Trifida:" + gh_token).encode("utf-8")
    ).decode("ascii")
    del gh_token

    # 上一次失败的 clone 可能留下临时目录。先改名保留，不删除任何文件。
    if STAGING_DIR.exists():
        if FAILED_DIR.exists():
            raise RuntimeError(
                f"发现旧临时目录 {STAGING_DIR}，且备份目录 {FAILED_DIR} 也已存在。"
                "为保护文件，停止操作，请先检查这两个目录。"
            )
        STAGING_DIR.rename(FAILED_DIR)
        print("已将上次失败的临时下载目录保留为：", FAILED_DIR)

    git_env = os.environ.copy()
    git_env["GIT_CONFIG_COUNT"] = "1"
    git_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    git_env["GIT_CONFIG_VALUE_0"] = auth_value
    del auth_value
    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", REPO_BRANCH, REPO_URL, str(STAGING_DIR)],
        env=git_env, text=True, capture_output=True
    )
    del git_env
    if result.returncode != 0:
        detail = result.stderr.strip() or result.stdout.strip() or "未提供详细错误信息"
        raise RuntimeError("GitHub 私有仓库克隆失败：\n" + detail)
    if not (STAGING_DIR / ".git").is_dir():
        raise RuntimeError("下载结束但临时目录中没有 Git 元数据；保留现场，不自动清理。")
    shutil.move(str(STAGING_DIR), str(REPO_DIR))
    print("项目代码获取完成：", REPO_DIR)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("项目分支：", REPO_BRANCH)
print("当前提交：", commit)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

## 06｜恢复已保存的 VocalParse 测试结果并生成乐谱文件

这里使用 GitHub 中已经保存的真实 25 秒 VocalParse 结构化样本，不重新运行模型、不下载权重。先检查依赖，再把 JSON、事件、MIDI 和 MusicXML 保存到 Google Drive 的独立新目录。

In [ ]:
import importlib.util

required_modules = ["music21", "pretty_midi"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError(
        "当前 Colab 缺少必要依赖：" + ", ".join(missing_modules) +
        "。本步骤不会自动安装软件；请把这条提示发回，我们再按当前环境补齐。"
    )

from musicmaster.core.vocalparse import (
    vocalparse_tokens_to_events,
    vocalparse_events_to_musicxml,
    musicxml_roundtrip_errors,
    vocalparse_lyric_alignment,
    musicxml_lyric_alignment,
)
import json
import hashlib
from datetime import datetime, timezone
import pretty_midi

fixture_path = REPO_DIR / "tests" / "fixtures" / "vocalparse_25s_sample.json"
if not fixture_path.is_file():
    raise FileNotFoundError(f"GitHub 中的历史测试样本未找到：{fixture_path}")

fixture_bytes = fixture_path.read_bytes()
parsed = json.loads(fixture_bytes.decode("utf-8"))
events = vocalparse_tokens_to_events(parsed)

bpm = float(parsed["bpm"])
note_events = [event for event in events if not event["is_rest"]]
rest_events = [event for event in events if event["is_rest"]]
displayed_lyrics = [
    item for item in vocalparse_lyric_alignment(events, bpm)
    if item["display_lyric"]
]

# 本样本的历史验证基准：50 个事件、43 个音符、7 个休止、39.5 拍、33 个歌词锚点。
assert len(events) == 50, f"事件数量异常：{len(events)}"
assert len(note_events) == 43, f"音符数量异常：{len(note_events)}"
assert len(rest_events) == 7, f"休止数量异常：{len(rest_events)}"
assert abs(sum(event["duration_beats"] for event in events) - 39.5) < 1e-7
assert len(displayed_lyrics) == 33, f"歌词锚点数量异常：{len(displayed_lyrics)}"

# 每次建立独立目录，不覆盖以前的任何测试结果。
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = MY_DRIVE / "XinsW_Lead_Transcription" / "VocalParse_Results" / f"restore_{run_id}"
output_dir.mkdir(parents=True, exist_ok=False)

parsed_path = output_dir / "vocalparse_parsed.json"
events_path = output_dir / "vocalparse_events.json"
musicxml_path = output_dir / "vocalparse.musicxml"
midi_path = output_dir / "vocalparse_lead.mid"
report_path = output_dir / "validation_report.json"

parsed_path.write_text(
    json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8"
)
events_path.write_text(
    json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8"
)

# 生成 MIDI：休止保留在事件 JSON / MusicXML 时间轴中，MIDI 只写入有音高的音符。
midi = pretty_midi.PrettyMIDI(initial_tempo=bpm)
instrument = pretty_midi.Instrument(program=0, name="VocalParse Lead")
seconds_per_beat = 60.0 / bpm
for event in note_events:
    instrument.notes.append(
        pretty_midi.Note(
            velocity=90,
            pitch=int(event["pitch_midi"]),
            start=float(event["start_beat"]) * seconds_per_beat,
            end=float(event["end_beat"]) * seconds_per_beat,
        )
    )
midi.instruments.append(instrument)
midi.write(str(midi_path))

# MusicXML 直接使用 beat-based events；随后按 tie-aware 规则做完整往返校验。
vocalparse_events_to_musicxml(events, bpm, musicxml_path)
roundtrip_errors = musicxml_roundtrip_errors(events, musicxml_path, bpm)
actual_lyrics = musicxml_lyric_alignment(musicxml_path)
expected_lyrics = [
    (item["start_beat"], item["lyric"])
    for item in displayed_lyrics
]
actual_lyric_pairs = [
    (item["start_beat"], item["lyric"])
    for item in actual_lyrics
]
assert not roundtrip_errors, "MusicXML 往返校验失败：" + "; ".join(roundtrip_errors)
assert actual_lyric_pairs == expected_lyrics, "MusicXML 歌词位置或内容与事件数据不一致"

# 重新读取 MIDI，确认实际保存的音符数量与时间范围。
midi_check = pretty_midi.PrettyMIDI(str(midi_path))
midi_note_count = sum(len(track.notes) for track in midi_check.instruments)
assert midi_note_count == 43, f"MIDI 重新读取后音符数异常：{midi_note_count}"

report = {
    "status": "passed",
    "source": "GitHub tests/fixtures/vocalparse_25s_sample.json",
    "source_sha256": hashlib.sha256(fixture_bytes).hexdigest(),
    "note": "恢复历史结构化样本；本次没有重新运行 VocalParse 模型推理。",
    "bpm": bpm,
    "event_count": len(events),
    "note_count": len(note_events),
    "rest_count": len(rest_events),
    "total_beats": sum(event["duration_beats"] for event in events),
    "displayed_lyric_count": len(displayed_lyrics),
    "musicxml_roundtrip_errors": roundtrip_errors,
    "musicxml_lyric_count": len(actual_lyrics),
    "midi_note_count_after_reload": midi_note_count,
    "artifacts": {
        "parsed_json": str(parsed_path),
        "events_json": str(events_path),
        "midi": str(midi_path),
        "musicxml": str(musicxml_path),
    },
}
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("历史 VocalParse 样本恢复与文件生成：通过")
print(json.dumps(report, ensure_ascii=False, indent=2))

## 07｜后续渲染模块

当前已包含结构化 JSON、事件 JSON、MIDI、MusicXML 的生成与校验。五线谱 SVG、简谱渲染将作为下一阶段接入；需要确认 Verovio、jianpu-ly 和 LilyPond 的可用性后再启用，不影响当前核心文件生成。